In [2]:
import pandas as pd
import numpy as np

In [3]:
win_task1 = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/windows_task1.csv")
win_task2 = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/windows_task2.csv")
win_task3 = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/windows_task3.csv")

In [9]:
# --- your mappings ---
task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

def decode_from_code(code: str):
    """
    Return dict with decoded fields from a code like 'C011', 'B112', or '*null'.
    """
    if not isinstance(code, str) or "null" in code.lower():
        return {
            "anim_family": np.nan,
            "anim_code": code if isinstance(code, str) else np.nan,
            "anim_type": "NO-ANIM",
            "anim_delay": np.nan,
            "anim_ease": np.nan,
            "anim_direction": np.nan,
            "block_anim": False,
        }

    fam = code[0]                 # 'A' / 'B' / 'C'
    spec = code[1:]               # e.g. '011' or '112'
    meta = animation_mapping.get(fam, {})

    # defaults
    delay = ease = direction = np.nan

    # parse indices safely
    try:
        i_delay = int(spec[0]) if len(spec) >= 1 else 0
        i_ease  = int(spec[1]) if len(spec) >= 2 else 0
        i_dir   = int(spec[2]) if len(spec) >= 3 else 0
    except ValueError:
        i_delay = i_ease = i_dir = 0

    # map delay/ease
    if "delay" in meta and isinstance(meta["delay"], list) and i_delay < len(meta["delay"]):
        delay = meta["delay"][i_delay]
    if "ease" in meta and isinstance(meta["ease"], list) and i_ease < len(meta["ease"]):
        ease = meta["ease"][i_ease]

    # map direction (B uses dict, A/C use list)
    if fam == "B" and isinstance(meta.get("direction"), dict):
        direction = meta["direction"].get(i_dir, np.nan)
    elif isinstance(meta.get("direction"), list) and i_dir < len(meta["direction"]):
        direction = meta["direction"][i_dir]

    return {
        "anim_family": fam,
        "anim_code": code,
        "anim_type": meta.get("type", np.nan),
        "delay": delay,
        "ease": ease,
        "direction": direction,
        "anim": True,
    }

def add_animation_columns(windows_df: pd.DataFrame) -> pd.DataFrame:
    """
    Make a copy of windows_df and add:
      anim_family, anim_code, anim_type, anim_delay, anim_ease, anim_direction, block_anim
    based on Task Id -> code mapping.
    """
    out = windows_df.copy()

    # ensure string task ids for mapping
    out["Task Id"] = out["Task Id"].astype(str).str.strip()

    # look up the code for each Task Id
    out["anim_code"] = out["Task Id"].map(task_mapping).fillna("Unknown")

    # decode each row
    decoded = out["anim_code"].apply(decode_from_code).apply(pd.Series)

    # attach
    for col in ["delay","ease","direction"]:
        out[col] = decoded[col]
    return out

windows_with_anim1 = add_animation_columns(win_task1)
windows_with_anim2 = add_animation_columns(win_task2)
windows_with_anim3 = add_animation_columns(win_task3)

windows_with_anim1.to_csv("outputs/windows_with_animation_details1.csv", index=False)
windows_with_anim2.to_csv("outputs/windows_with_animation_details2.csv", index=False)
windows_with_anim3.to_csv("outputs/windows_with_animation_details3.csv", index=False)
